# TP NLP — Prétraitement, NER, POS et Word2Vec

Ce notebook couvre :
- Le prétraitement de texte (minuscules, tokenisation, ponctuation, stopwords, lemmatisation)
- La reconnaissance d'entités nommées (**NER**) avec spaCy
- L'étiquetage morphosyntaxique (**POS**) avec NLTK
- La vectorisation avec **Word2Vec** (gensim) et sa visualisation


## 0. Installation et imports

In [ ]:
# Sur Google Colab, décommentez si nécessaire :
# !pip install -q spacy nltk gensim
# !python -m spacy download en_core_web_sm

In [ ]:
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import spacy
from gensim.models import Word2Vec

# Téléchargement des ressources NLTK
for res in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'tagsets']:
    nltk.download(res, quiet=True)

# Chargement du modèle spaCy anglais
nlp = spacy.load('en_core_web_sm')
print("Environnement prêt ✅")

## 1. Jeu de données

In [ ]:
data = {
    'Review': [
        'At McDonald\'s the food was ok and the service was bad.',
        'I would not recommend this Japanese restaurant to anyone.',
        'I loved this restaurant when I traveled to Thailand last summer.',
        'The menu of Loving has a wide variety of options.',
        'The staff was friendly and helpful at Google\'s employees restaurant.',
        'The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.',
        'I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.',
        'The sushi at Sushi Express is always fresh and flavorful.',
        'The steakhouse on Main Street has a cozy atmosphere and excellent steaks.',
        'The dessert selection at Sweet Treats is to die for!'
    ]
}

df_raw = pd.DataFrame(data)   # données BRUTES (on les conserve)
df_raw

# Exercice 1 — Prétraitement, NER et POS

## 1.1 Fonction `preprocess_text()`

Elle convertit en minuscules, tokenise, retire la ponctuation, retire les mots vides (stopwords), applique un lemmatiseur, puis renvoie les chaînes nettoyées.

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(reviews):
    """Reçoit une liste de textes et renvoie une liste de chaînes prétraitées."""
    cleaned = []
    for text in reviews:
        # 1) minuscules + tokenisation
        tokens = word_tokenize(text.lower())
        # 2) suppression de la ponctuation
        tokens = [t for t in tokens if t not in string.punctuation]
        # 3) suppression des mots vides
        tokens = [t for t in tokens if t not in stop_words]
        # 4) lemmatisation
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
        # 5) reconstruction de la chaîne
        cleaned.append(' '.join(tokens))
    return cleaned

# Application et vérification
cleaned_reviews = preprocess_text(df_raw['Review'])
for original, clean in zip(df_raw['Review'], cleaned_reviews):
    print(f"BRUT   : {original}")
    print(f"NETTOYÉ: {clean}\n")

## 1.2 Nouveau jeu de données avec le texte nettoyé

On conserve **deux** DataFrames : `df_raw` (brut) et `df_clean` (prétraité).

In [ ]:
df_clean = df_raw.copy()
df_clean['Review'] = cleaned_reviews
df_clean

## 1.3 Fonction `perform_ner()`

Utilise spaCy (`en_core_web_sm`) pour extraire les entités nommées et leurs étiquettes (ex. `ORG`, `GPE`, `DATE`).

In [ ]:
def perform_ner(text):
    """Renvoie la liste des (entité, étiquette) trouvées dans le texte."""
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

# Application sur les données BRUTES (le NER fonctionne mieux sur le texte non nettoyé)
print("=== NER sur données BRUTES ===")
for text in df_raw['Review']:
    entities = perform_ner(text)
    print(f"{text}\n  -> {entities}\n")

## 1.4 Fonction `perform_pos_tagging()`

Utilise `nltk.pos_tag` pour attribuer une étiquette morphosyntaxique à chaque token.

In [ ]:
def perform_pos_tagging(text):
    """Tokenise le texte et renvoie la liste des (mot, étiquette POS)."""
    tokens = word_tokenize(text)
    return nltk.pos_tag(tokens)

# Vérification rapide
perform_pos_tagging(df_raw['Review'][0])

## 1.5 Application aux données brutes ET prétraitées + analyse

On compare l'étiquetage POS sur les deux versions.

In [ ]:
print("=== POS — données BRUTES ===\n")
for text in df_raw['Review']:
    print(text)
    print(perform_pos_tagging(text), "\n")

In [ ]:
print("=== POS — données PRÉTRAITÉES ===\n")
for text in df_clean['Review']:
    print(text)
    print(perform_pos_tagging(text), "\n")

In [ ]:
# Signification des étiquettes POS (Penn Treebank)
nltk.download('tagsets', quiet=True)
for tag in ['NN', 'NNP', 'JJ', 'VBD', 'DT', 'IN']:
    nltk.help.upenn_tagset(tag)

### 🔎 Analyse (Exercice 1)

- Sur le texte **brut**, le NER identifie correctement les organisations (`ORG` : McDonald's, Google, Pizza Hut, Bella Italia…), les lieux (`GPE` : Thailand) et les dates (`DATE` : last summer). Le contexte, les majuscules et la structure grammaticale aident spaCy.
- Sur le texte **prétraité**, le NER et le POS sont **beaucoup moins fiables** : la mise en minuscules supprime les majuscules qui signalent les noms propres, et la suppression des stopwords casse la structure syntaxique nécessaire au POS tagging.
- **Conclusion** : NER et POS s'appliquent de préférence sur le texte **brut** ; le prétraitement agressif est plutôt utile pour la vectorisation (Word2Vec, TF-IDF…).

# Exercice 2 — Visualisation des plongements lexicaux (Word2Vec)

## 2.1 Entraînement du modèle Word2Vec

On utilise les données **prétraitées et tokenisées**.

In [ ]:
# Tokenisation des reviews nettoyées (liste de listes de mots)
tokenized_reviews = [review.split() for review in df_clean['Review']]
print(tokenized_reviews[:3])

In [ ]:
word2vec_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=100,   # dimension de chaque vecteur
    window=5,          # taille de la fenêtre de contexte
    min_count=1,       # inclut les mots apparaissant au moins 1 fois
    workers=1,
    seed=42
)

print("Nombre de mots dans le vocabulaire :", len(word2vec_model.wv.index_to_key))
print("Dimension d'un vecteur de mot       :", word2vec_model.wv.vector_size)
print("Forme de la matrice d'embeddings    :", word2vec_model.wv.vectors.shape)

# Exemple : vecteur du mot 'restaurant'
print("\nVecteur de 'restaurant' (5 premières valeurs) :")
print(word2vec_model.wv['restaurant'][:5])

**Dimensions de l'objet Word2Vec**

- La matrice d'embeddings a la forme `(taille_du_vocabulaire, vector_size)`, ici `(N, 100)`.
- Chaque mot est représenté par un vecteur de **100 dimensions**.
- Ces 100 dimensions sont des caractéristiques **latentes** apprises : elles ne correspondent pas à des concepts interprétables individuellement, mais leur combinaison encode le contexte d'apparition des mots. Des mots utilisés dans des contextes similaires ont des vecteurs proches.

## 2.2 Fonction `plot_word_embeddings()`

Comme les vecteurs ont 100 dimensions, on les réduit à **2D** (PCA) pour pouvoir les tracer dans un nuage de points, puis on annote chaque point avec le mot correspondant.

In [ ]:
from sklearn.decomposition import PCA

def plot_word_embeddings(model):
    """Trace les plongements lexicaux d'un modèle Word2Vec dans un nuage de points 2D."""
    words = list(model.wv.index_to_key)
    vectors = model.wv[words]

    # Réduction 100D -> 2D
    coords = PCA(n_components=2, random_state=42).fit_transform(vectors)

    plt.figure(figsize=(14, 10))
    plt.scatter(coords[:, 0], coords[:, 1], c='steelblue', edgecolors='k', s=60)

    # Annotation de chaque point avec le mot
    for word, (x, y) in zip(words, coords):
        plt.annotate(word, xy=(x, y), xytext=(5, 2),
                     textcoords='offset points', fontsize=10)

    plt.title("Plongements lexicaux Word2Vec (projection PCA 2D)")
    plt.xlabel("Composante 1")
    plt.ylabel("Composante 2")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

plot_word_embeddings(word2vec_model)

### 🔎 Analyse (Exercice 2)

**Les mots apparentés sont-ils proches les uns des autres ?**

En pratique, sur ce jeu de données, les regroupements sémantiques sont **faibles voire peu fiables**.

**Raisons possibles :**
1. **Corpus minuscule** : seulement 10 phrases. Word2Vec a besoin de dizaines de milliers de phrases pour apprendre des relations sémantiques robustes.
2. **`min_count=1`** : des mots vus une seule fois donnent des vecteurs quasi aléatoires (peu de signal de contexte).
3. **`vector_size=100` trop grand** pour si peu de données : le modèle sur-paramètre.
4. **Perte d'information due à la PCA** : la projection 100D → 2D ne conserve qu'une partie de la variance.

**Pistes d'amélioration :**
- Utiliser un corpus beaucoup plus grand, ou un modèle **pré-entraîné** (Google News, GloVe).
- Réduire `vector_size` (ex. 20–50) et augmenter le nombre d'`epochs`.
- Tester une visualisation **t-SNE** ou **UMAP**, souvent plus parlantes que la PCA pour les embeddings.

## 2.3 (Bonus) Améliorations : t-SNE + paramètres ajustés

In [ ]:
from sklearn.manifold import TSNE

# Modèle affiné : vecteurs plus petits, plus d'epochs
w2v_tuned = Word2Vec(sentences=tokenized_reviews, vector_size=30, window=5,
                     min_count=1, workers=1, seed=42, epochs=200)

words = list(w2v_tuned.wv.index_to_key)
vecs = w2v_tuned.wv[words]

# t-SNE (perplexity doit être < nombre d'échantillons)
perp = min(15, max(2, len(words) - 1))
coords = TSNE(n_components=2, random_state=42, perplexity=perp, init='pca').fit_transform(vecs)

plt.figure(figsize=(14, 10))
plt.scatter(coords[:, 0], coords[:, 1], c='darkorange', edgecolors='k', s=60)
for w, (x, y) in zip(words, coords):
    plt.annotate(w, (x, y), xytext=(5, 2), textcoords='offset points', fontsize=10)
plt.title("Plongements Word2Vec affinés — projection t-SNE")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Exemple : mots les plus similaires à 'restaurant'
print("Mots les plus proches de 'restaurant' :")
print(w2v_tuned.wv.most_similar('restaurant', topn=5))